# Explore two CIViC bundles

This notebook uses two small CIViC examples to show how the Starter Kit loads and explores GKM Bundles. You will inspect their contents, work with typed GKM objects, and follow relationships between objects.

A **bundle** is one GKM Bundle loaded by the Starter Kit. Within a bundle, a **collection** is a named mapping from object identifiers to objects. Producers use collections to organize objects by purpose or type. A **bundle format** defines the producer-specific structure, including its collections, keys, references, and provenance. A separate JSON Schema describes that format.

- **CIViC AID 9**: a Tier II clinical-significance assertion under the AMP/ASCO/CAP Guidelines (2017).
- **CIViC AID 251**: a likely-oncogenic assertion under the ClinGen/CGC/VICC Guidelines for Oncogenicity (2022).

Both files use referenced JSON. Each shared GKM object appears once, and relationships point to it by a path within the same file.

## 1. Import the library

The public API starts in `gkm.starter`.

In [1]:
import json
from pathlib import Path

from gkm import starter

## 2. Locate the data and schema

Each example consists of bundle data plus the schema for the CIViC bundle format:

* `civic-aid-9-bundle.json` contains the data for CIViC AID 9.
* `civic-aid-251-bundle.json` contains the data for CIViC AID 251.
* `civic-gks-bundle-v0.1.0.schema.json` defines the CIViC-specific structure and identifies the GKM schemas used by its collections.

The two bundles use the same format, so they share one schema. The path setup works when Jupyter starts either at the repository root or in `notebooks/civic`.

In [2]:
bundle_dir = Path("bundles")
if not bundle_dir.is_dir():
    bundle_dir = Path("notebooks/civic/bundles")

if not bundle_dir.is_dir():
    message = "Run this notebook from the repository root or notebooks/civic"
    raise FileNotFoundError(message)

civic_aid_9_name = "civic-aid-9"
civic_aid_251_name = "civic-aid-251"

aid_9_path = bundle_dir / f"{civic_aid_9_name}-bundle.json"
aid_251_path = bundle_dir / f"{civic_aid_251_name}-bundle.json"
schema_path = bundle_dir / "civic-gks-bundle-v0.1.0.schema.json"

### Check GKM version compatibility

Before loading data, check that the schema references the same GKM product versions as the installed Python libraries. This is a compatibility check, not full JSON Schema validation. `load_bundle(..., schema=...)` performs the same check automatically; calling it directly makes this important boundary visible.

In [3]:
with schema_path.open(encoding="utf-8") as stream:
    civic_schema = json.load(stream)

starter.check_gkm_version_compatibility(civic_schema)
starter.supported_gkm_versions()

{'gks-core': '1.1.0',
 'vrs': '2.1.0-snapshot.2026-02.2',
 'cat-vrs': '1.1.0-snapshot.2026-02.3',
 'va-spec': '1.1.0-snapshot.2026-06.1'}

## 3. Register and load the bundles

The bundle registry associates each data file with its schema, producer, and a short name. This lets later calls load a bundle by name instead of repeating its paths. `replace=True` makes the registration cells safe to run again during an interactive session.

In [4]:
producer = "CIViC"

starter.bundles.register(
    starter.bundles.BundleRegistration(
        name=civic_aid_9_name,
        source=aid_9_path,
        schema=schema_path,
        producer=producer,
    ),
    replace=True,
)
starter.bundles.register(
    starter.bundles.BundleRegistration(
        name=civic_aid_251_name,
        source=aid_251_path,
        schema=schema_path,
        producer=producer,
    ),
    replace=True,
)

Now load each registered bundle. `load_bundle()` reads the producer-defined collections and constructs supported GKM objects with their reference models. The schema also lets the loader reject bundles that target incompatible GKM versions.

In [5]:
aid_9 = starter.load_bundle(civic_aid_9_name)
aid_251 = starter.load_bundle(civic_aid_251_name)
aid_9, aid_251

(Bundle(name='civic-aid-9', collections=17),
 Bundle(name='civic-aid-251', collections=17))

## 4. Inspect the collections

The Starter Kit preserves the top-level collection names in the loaded bundle instead of requiring a universal set of names. For example, CIViC's `sequenceReference` collection maps sequence identifiers to sequence-reference objects. Names such as `sequenceReference`, `proposition`, and `assertion` are defined by the CIViC bundle format.

Start with `collection_names()` to discover which collections a loaded bundle exposes. Names are returned in the same order as the source document.

In [6]:
aid_9.collection_names()

('sequenceReference',
 'location',
 'variant',
 'feature',
 'molecularProfile',
 'disease',
 'phenotype',
 'conditionSet',
 'therapy',
 'therapyGroup',
 'variantOrigin',
 'source',
 'method',
 'organization',
 'proposition',
 'evidence',
 'assertion')

### Access a collection

A loaded bundle creates collection accessors from its top-level collection names. The following expressions all return the same collection:

* `aid_9.sequenceReference` is convenient when the name is known in advance.
* `aid_9["sequenceReference"]` uses standard mapping syntax.
* `aid_9.collection("sequenceReference")` is explicit and works well when the name is stored in a variable.

These are lookups against the loaded bundle. They are not hardcoded Python properties.

In [7]:
sequence_references = aid_9.sequenceReference

(
    sequence_references is aid_9["sequenceReference"],
    sequence_references is aid_9.collection("sequenceReference"),
)

(True, True)

An unrecognized attribute is not guessed or silently created. It raises `AttributeError`, as a misspelled collection name demonstrates:

In [8]:
try:
    aid_9.sequenceReferences  # noqa: B018
except AttributeError as error:
    print(f"Unrecognized collection: {error}")

Unrecognized collection: sequenceReferences


## 5. Work with a typed GKM object

When an object can be validated independently, the Starter Kit constructs it with the applicable GKM reference model. This provides a known type, validation, and attribute access instead of leaving every value as a raw dictionary.

The example below retrieves a VRS `SequenceReference` by its collection key. The collection accessor selects the collection, and the identifier inside brackets selects one object from it.

In [9]:
sequence_id = "SQ.6CnHhDq_bDCsuIBf0AzxtKq_lXYM7f0m"
sequence_reference = sequence_references[sequence_id]

sequence_reference

SequenceReference(id=None, type='SequenceReference', name=None, description=None, aliases=None, extensions=None, refgetAccession='SQ.6CnHhDq_bDCsuIBf0AzxtKq_lXYM7f0m', residueAlphabet=None, circular=None, sequence=None, moleculeType=None)

## 6. Resolve a local reference

The assertion does not contain a second copy of its proposition. Instead, its `proposition` field contains a local JSON Pointer such as `#/proposition/...`, which identifies a path within the bundle. `dereference()` follows that pointer and returns the referenced object. `resolve()` is its plain-language alias.

In [10]:
assertion_9 = aid_9.assertion["civic.aid:9"]
proposition_9 = aid_9.resolve(assertion_9["proposition"])

assertion_9["proposition"], proposition_9.model_dump(exclude_none=True)

('#/proposition/civic.proposition:f0SrtLbW05PqfqLs-hOK4tZxI3xO3kMO',
 {'id': 'civic.proposition:f0SrtLbW05PqfqLs-hOK4tZxI3xO3kMO',
  'type': 'VariantClinicalSignificanceProposition',
  'subjectVariant': '#/molecularProfile/civic.mpid:1594',
  'geneContextQualifier': '#/feature/civic.gid:154',
  'alleleOriginQualifier': '#/variantOrigin/civic.variantOrigin:SOMATIC',
  'predicate': 'hasClinicalSignificanceFor',
  'objectCondition': '#/disease/civic.did:2950'})

## 7. Dereference everything below the proposition

Following one pointer at a time is useful when you need a specific object. `dereference()` is useful when you need a self-contained view of a complete object graph. It replaces every reachable local pointer with inline data.

Here, the result contains the proposition and everything reachable below it, not the assertion or the complete bundle. A pointer that would create a cycle remains a pointer because JSON cannot represent infinitely nested data.

In [11]:
inline_proposition_9 = aid_9.dereference(proposition_9)
inline_proposition_9

{'id': 'civic.proposition:f0SrtLbW05PqfqLs-hOK4tZxI3xO3kMO',
 'type': 'VariantClinicalSignificanceProposition',
 'subjectVariant': {'id': 'civic.mpid:1594',
  'type': 'CategoricalVariant',
  'name': 'ACVR1 G328V',
  'aliases': ['GLY328VAL'],
  'extensions': [{'name': 'molecularProfileScore', 'value': 35.0},
   {'name': 'hgvsDescriptions',
    'value': ['NM_001105.4:c.983G>T',
     'NP_001096.1:p.Gly328Val',
     'NC_000002.11:g.158622516C>A',
     'ENST00000434821.1:c.983G>T',
     'NC_000002.12:g.157766004C>A',
     'NM_001111067.4:c.983G>T',
     'NP_001104537.1:p.Gly328Val',
     'ENST00000434821.7:c.983G>T',
     'ENSP00000405004.1:p.Gly328Val']},
   {'name': 'maneSelectTranscript', 'value': 'ENST00000434821.7:c.983G>T'},
   {'name': 'representativeVariantCoordinates',
    'value': {'chromosome': '2',
     'start': 158622516,
     'stop': 158622516,
     'reference_bases': 'C',
     'variant_bases': 'A',
     'ensembl_version': 75,
     'representative_transcript': 'ENST00000434821

## 8. Compare the assertion models

The two bundles use the same CIViC format, but their assertions answer different scientific questions. A compact summary highlights their different VA-Spec proposition types, predicates, and classification systems.

In [12]:
def assertion_summary(
    bundle: starter.bundles.Bundle, assertion_id: str
) -> dict[str, str]:
    """Summarize one assertion and its referenced proposition.

    :param bundle: Bundle containing the assertion.
    :param assertion_id: Collection key for the assertion.
    :return: Selected assertion, classification, and proposition fields.
    """
    assertion = bundle.assertion[assertion_id]
    proposition = bundle.resolve(assertion["proposition"])
    coding = assertion["classification"]["primaryCoding"]
    return {
        "assertion": assertion_id,
        "classification_system": coding["system"],
        "classification": coding["code"],
        "proposition_type": proposition.type,
        "predicate": proposition.predicate,
    }


[
    assertion_summary(aid_9, "civic.aid:9"),
    assertion_summary(aid_251, "civic.aid:251"),
]

[{'assertion': 'civic.aid:9',
  'classification_system': 'AMP/ASCO/CAP Guidelines, 2017',
  'classification': 'tier ii',
  'proposition_type': 'VariantClinicalSignificanceProposition',
  'predicate': 'hasClinicalSignificanceFor'},
 {'assertion': 'civic.aid:251',
  'classification_system': 'ClinGen/CGC/VICC Guidelines for Oncogenicity, 2022',
  'classification': 'likely oncogenic',
  'proposition_type': 'VariantOncogenicityProposition',
  'predicate': 'isOncogenicFor'}]

## Why this matters

The Starter Kit turns CIViC's referenced JSON into a discoverable Python interface. A consumer can find the available collections, work with typed GKM objects, and follow relationships without writing reference-resolution code.

The same approach is available to other data producers. A producer can model its knowledge with GKM standards, organize it in a schema-defined bundle format, and let consumers use the same Starter Kit workflow without copying CIViC's format.